In [14]:
import os
from getpass import getpass
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
# langchain library has been restructured recently. Use langchain_classic for older imports as mentioned below.
from langchain_classic.chains import RetrievalQA

In [17]:
os.environ["GROQ_API_KEY"] = getpass("Enter you Groq API Key: ")

In [18]:
# configuration
vector_db_path = "/Users/arun/Jupyter Projects RAG/6.2. RAG with LangChain/vector_db"
collection_name = "document_collection"

In [4]:
# loading the embedding model - default model
embedding = HuggingFaceEmbeddings()

W1124 18:36:32.747000 16433 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
# initialize groq llm
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0
)

In [9]:
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embedding,
    persist_directory=vector_db_path
)

In [10]:
retriever = vector_store.as_retriever()

In [11]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [19]:
query = "What does the document say about abstract classes being inherited?"
response = qa_chain.invoke({"query":query})
print(response["result"])

The document explains that **abstract classes participate in inheritance just like any other class, but the presence of abstract members imposes extra rules on their subclasses**. The key points are:

1. **Abstract members are inherited**  
   - Any instance method (declared or inherited) of a class *C* that is abstract remains abstract in *C* unless it is overridden with a concrete implementation.  
   - If a superclass declares an abstract method with package‑private access, that abstract method is also inherited by the subclass. (See the paragraph that begins “Any of the member methods (§8.2) of C – either declared or inherited – is abstract.”)

2. **Subclasses must either implement or remain abstract**  
   - When a class inherits one or more abstract methods, it **must** either provide concrete implementations for all of them or be declared `abstract` itself.  
   - Example 8.1.1.1‑1 shows this:  
     ```java
     abstract class Point { … abstract void alert(); }
     abstract cl